In [ ]:
%reset -f
# Pick up edits to physics/, model/ and notebooks/ without restarting the kernel.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT:", REPO_ROOT)

import mplhep as hep
import numpy as np
import pandas as pd
import torch
import yaml
from matplotlib import pyplot as plt
from scipy.stats import ks_2samp

from model import LightningWBoson
from physics.diagnostics import component_calibration
from physics.physics import (
    TOR,
    W_MASS,
    dphi,
    four_vector_pairs,
    invariant_mass,
    invariant_mass2,
    pt,
    sphi,
)
from physics.selection import JET_ENERGY_COLUMNS, lepton_p4, selection_masks
from physics.torchBoost import Booster as TorchBooster
from train import train
from notebooks.plottingtool import (
    plot_1d_hist,
    plot_2d_hist,
    plot_angular_1d_grid,
    plot_angular_2d_grid,
    plot_gradient_cosine_heatmaps,
    plot_gradient_norms,
    plot_loss_curves,
    plot_pair,
)
hep.style.use("ATLAS")

In [ ]:
CONFIG_PATH = REPO_ROOT / "configs/kfold_config.yaml"
# CONFIG_PATH = REPO_ROOT / "configs/config.yaml"
# CONFIG_PATH = REPO_ROOT / "outputs/sweep-2026-09-18/configs/W_d216_b8192_w20.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text())

# "latest" prefers last.ckpt; "best" reads the monitored metric out of the filename.
CKPT_SELECTION = "best"

RUN_DIR = Path(cfg["paths"]["saved_path"]).expanduser()
if not RUN_DIR.is_absolute():
    RUN_DIR = (REPO_ROOT / RUN_DIR).resolve()

# Cross-fitting runs write one model per fold under saved_path/fold<i>; a plain
# single-model run writes saved_path/logs directly.
LOG_ROOT = next(iter(sorted(RUN_DIR.glob("fold0"))), RUN_DIR)

ckpt_files = sorted(
    (LOG_ROOT / "logs").glob("version_*/checkpoints/*.ckpt"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if not ckpt_files:
    raise FileNotFoundError(f"No checkpoint files found under {LOG_ROOT}")
epoch_ckpts = [path for path in ckpt_files if path.name != "last.ckpt"] or ckpt_files


def best_ckpt_by_metric(paths, cfg):
    """Checkpoint with the best monitored metric in its filename, or None if unlabelled."""
    trainer_cfg = cfg.get("trainer", {})
    mode = trainer_cfg.get("monitor_mode", "min")
    monitored = trainer_cfg.get("monitor_metric", "val_static_loss")

    for metric in [monitored, "val_static_loss", "val_loss"]:
        token = f"{metric}="
        scored = []
        for path in paths:
            if token not in path.stem:
                continue
            try:
                scored.append((path, float(path.stem.rsplit(token, 1)[1].split("-", 1)[0])))
            except ValueError:
                pass
        if scored:
            path, value = sorted(scored, key=lambda item: item[1], reverse=(mode == "max"))[0]
            return path, f"best by {metric} ({value:.4g})"
    return None


if CKPT_SELECTION == "latest":
    ckpt_path = next((path for path in ckpt_files if path.name == "last.ckpt"), epoch_ckpts[0])
    reason = "most recent"
elif CKPT_SELECTION == "best":
    # Falls back to the newest epoch checkpoint when no filename carries the metric.
    ckpt_path, reason = best_ckpt_by_metric(epoch_ckpts, cfg) or (epoch_ckpts[0], "most recent")
else:
    raise ValueError(f"CKPT_SELECTION must be 'best' or 'latest', got {CKPT_SELECTION!r}")

LOG_DIR = ckpt_path.parents[1]
print(f"Run directory: {LOG_ROOT} ({len(ckpt_files)} checkpoints found)")
print(f"Using {reason} checkpoint: {ckpt_path}")
print(f"Using log directory: {LOG_DIR}")

In [ ]:
METRICS_PATH = LOG_DIR / "metrics.csv"
metrics = pd.read_csv(METRICS_PATH)

loss_diagnostics = plot_loss_curves(METRICS_PATH, cfg)
plot_gradient_cosine_heatmaps(metrics)
gradient_norms = plot_gradient_norms(metrics)

In [ ]:
# Reuse the data split defined by the training config.
dm = train.main(train=False, config_path=str(CONFIG_PATH))

model = LightningWBoson.load_from_checkpoint(str(ckpt_path), weights_only=False, strict=False)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Inference covers the whole test set so cuts can be retuned without re-running it.
feature_batches = []
prediction_batches = []
target_batches = []

with torch.no_grad():
    for inputs, targets in dm.test_dataloader():
        inputs = inputs.to(device)
        feature_batches.append(inputs.cpu().numpy())
        prediction_batches.append(model(inputs).cpu().numpy())
        target_batches.append(targets.numpy())

all_features = np.concatenate(feature_batches, axis=0)
all_predictions = np.concatenate(prediction_batches, axis=0)
all_true_labels = np.concatenate(target_batches, axis=0)
print(f"Evaluated {len(all_predictions)} test events on {device}.")

In [ ]:
# Cut thresholds live in physics/selection.py so they stay unit tested.
NJETS_SELECTION = None  # 0 / 1 / 2 keeps only that jet multiplicity; None keeps every event

masks = selection_masks(all_features, njets=NJETS_SELECTION)

# No cut is applied by default; narrow the sample with e.g. selection &= masks["met"].
selection = np.ones(len(all_features), dtype=bool)

features = all_features[selection]
predictions = all_predictions[selection]
true_labels = all_true_labels[selection]
print(
    f"Selected {len(features)} / {len(all_features)} events "
    f"(njets selection: {NJETS_SELECTION})."
)

In [ ]:
CHARGES = ["+", "-"]

# Leptons, W bosons and the targets, as (events, {+, -}, {px, py, pz, E}).
lep_p4 = lepton_p4(features)
pred_w_p4 = four_vector_pairs(predictions[:, 0:8])
true_w_p4 = four_vector_pairs(true_labels[:, 0:8])
true_w_mass = np.abs(true_labels[:, 8:10])  # the targets carry the W masses separately

# Each neutrino is its W minus the measured lepton; the Higgs is the W pair.
pred_nu_p4 = pred_w_p4 - lep_p4
true_nu_p4 = true_w_p4 - lep_p4
pred_higgs_p4 = pred_w_p4.sum(axis=1)
true_higgs_p4 = true_w_p4.sum(axis=1)
pred_higgs_mass = invariant_mass(pred_higgs_p4)
true_higgs_mass = invariant_mass(true_higgs_p4)

In [ ]:
bins_edges = np.linspace(0, 200, 51)
for column, label in zip(JET_ENERGY_COLUMNS, ["j0", "j1"]):
    plt.hist(features[:, column], bins=bins_edges, histtype="step", label=label)
plt.xscale("log")
plt.xlim(10, 200)
plt.ylim(0, 5e3)
plt.xlabel(r"$E_{j}$ [GeV]")
plt.legend()
plt.show()

In [ ]:
component_names = [f"W{charge} {axis}" for charge in CHARGES for axis in ["px", "py", "pz", "E"]]
calibration = component_calibration(predictions[:, :8], true_labels[:, :8])

rows = []
for index, name in enumerate(component_names):
    pred = predictions[:, index]
    truth = true_labels[:, index]
    valid = np.isfinite(pred) & np.isfinite(truth)
    rows.append(
        {
            "component": name,
            "pred_mean": pred[valid].mean() if valid.any() else np.nan,
            "truth_mean": truth[valid].mean() if valid.any() else np.nan,
            **{metric: values[index] for metric, values in calibration.items()},
        }
    )

fourvec_diagnostics = pd.DataFrame(rows).set_index("component")
display(fourvec_diagnostics.round(4))

In [ ]:
print(
    f"Mean predicted Higgs mass: {pred_higgs_mass.mean():.2f} GeV, "
    f"Std: {pred_higgs_mass.std():.2f} GeV"
)

higgs_observables = [
    (
        pt(pred_higgs_p4[:, 0], pred_higgs_p4[:, 1]),
        pt(true_higgs_p4[:, 0], true_higgs_p4[:, 1]),
        r"$p_T^{H}$",
        np.linspace(0, 600, 51),
    ),
    (pred_higgs_p4[:, 2], true_higgs_p4[:, 2], r"$p_z^{H}$", np.linspace(-600, 600, 51)),
    (pred_higgs_p4[:, 3], true_higgs_p4[:, 3], r"$E_H$", np.linspace(100, 600, 51)),
    (pred_higgs_mass, true_higgs_mass, r"$m_H$", np.linspace(124.8, 125.2, 51)),
]
for pred, truth, name, bins in higgs_observables:
    plot_pair(pred, truth, name, bins)

In [ ]:
w_components = [
    (0, "p_x", np.linspace(-150, 150, 51)),
    (1, "p_y", np.linspace(-150, 150, 51)),
    (2, "p_z", np.linspace(-300, 300, 51)),
    (3, "E", np.linspace(0, 300, 51)),
]

for component, symbol, bins in w_components:
    for index, charge in enumerate(CHARGES):
        plot_pair(
            pred_w_p4[:, index, component],
            true_w_p4[:, index, component],
            rf"${symbol}^{{W^{charge}}}$",
            bins,
        )

for index, charge in enumerate(CHARGES):
    plot_pair(
        invariant_mass(pred_w_p4[:, index]),
        true_w_mass[:, index],
        rf"$m_{{W^{charge}}}$",
        np.linspace(0, 100, 51),
    )

In [ ]:
def split_by_mask(p4_pair, mask):
    """Split an (events, 2, 4) pair into (selected, other) with a per-event mask."""
    first, second = p4_pair[:, 0], p4_pair[:, 1]
    mask = mask[:, np.newaxis]
    return np.where(mask, first, second), np.where(mask, second, first)


def alpha_func(lep_on_p4, neu_on_p4, lep_off_p4, neu_off_p4):
    """Momentum fraction of the neutrino from the heavier lepton + di-neutrino system."""
    dinu_p4 = neu_on_p4 + neu_off_p4
    norm_on = np.linalg.norm(neu_on_p4[:, :3], axis=-1)
    norm_off = np.linalg.norm(neu_off_p4[:, :3], axis=-1)
    on_is_heavier = invariant_mass(lep_on_p4 + dinu_p4) > invariant_mass(lep_off_p4 + dinu_p4)
    return np.where(on_is_heavier, norm_on, norm_off) / (norm_on + norm_off + TOR)


true_on_first = np.abs(true_w_mass[:, 0] ** 2 - W_MASS**2) < np.abs(
    true_w_mass[:, 1] ** 2 - W_MASS**2
)
pred_on_first = np.abs(invariant_mass2(pred_w_p4[:, 0]) - W_MASS**2) < np.abs(
    invariant_mass2(pred_w_p4[:, 1]) - W_MASS**2
)

# The lepton assignment always comes from truth, so pred and truth alpha stay comparable.
lep_on_p4, lep_off_p4 = split_by_mask(lep_p4, true_on_first)
true_w_on_p4, true_w_off_p4 = split_by_mask(true_w_p4, true_on_first)
pred_w_on_p4, pred_w_off_p4 = split_by_mask(pred_w_p4, pred_on_first)

true_alpha = alpha_func(lep_on_p4, true_w_on_p4 - lep_on_p4, lep_off_p4, true_w_off_p4 - lep_off_p4)
pred_alpha = alpha_func(lep_on_p4, pred_w_on_p4 - lep_on_p4, lep_off_p4, pred_w_off_p4 - lep_off_p4)

plot_pair(pred_alpha, true_alpha, r"$\alpha$", np.linspace(0, 1, 51), unit="null", vmax=1e3)

In [ ]:
def angular_features_like_training(lep_p4, true_w_p4, pred_w_p4, device):
    """Lepton (theta, phi) per W rest frame, keeping only events with a valid boost."""
    lep = torch.as_tensor(lep_p4, dtype=torch.float32, device=device)
    true_w = torch.as_tensor(true_w_p4, dtype=torch.float32, device=device)
    pred_w = torch.as_tensor(pred_w_p4, dtype=torch.float32, device=device)

    true_booster = TorchBooster(lep, true_w)
    pred_booster = TorchBooster(lep, pred_w)
    valid = true_booster.valid_rest_frame_mask() & pred_booster.valid_rest_frame_mask()

    true_ang = torch.stack(true_booster.lep_theta_phi_in_w_rest(), dim=-1)[valid]
    pred_ang = torch.stack(pred_booster.lep_theta_phi_in_w_rest(), dim=-1)[valid]
    return true_ang.cpu().numpy(), pred_ang.cpu().numpy(), valid.cpu().numpy()


true_ang, pred_ang, angular_valid = angular_features_like_training(
    features[:, :8], true_labels[:, :8], predictions[:, :8], device
)
print(f"Angular valid events: {angular_valid.sum()} / {len(angular_valid)}")

# Columns 0..3 are (theta+, phi+, theta-, phi-); the booster already wraps phi to (-pi, pi].
true_theta, true_phi = true_ang[:, [0, 2]], true_ang[:, [1, 3]]
pred_theta, pred_phi = pred_ang[:, [0, 2]], pred_ang[:, [1, 3]]

In [ ]:
for index, charge in enumerate(CHARGES):
    plot_pair(
        pred_theta[:, index] / np.pi,
        true_theta[:, index] / np.pi,
        rf"$\theta^\ast_{{\ell^{charge}}}$",
        np.linspace(0, 1, 51),
        unit=r"rad/$\pi$",
        vmax=8e2,
    )

for index, charge in enumerate(CHARGES):
    plot_pair(
        pred_phi[:, index] / np.pi,
        true_phi[:, index] / np.pi,
        rf"$\phi^\ast_{{\ell^{charge}}}$",
        np.linspace(-1, 1, 61),
        unit=r"rad/$\pi$",
        vmax=2e2,
    )

In [ ]:
def observable(pred, truth, label, bins, log=True, vmax=8e2):
    return {"pred": pred, "truth": truth, "label": label, "bins": bins, "log": log, "vmax": vmax}


angular_observables = [
    observable(
        pred_theta.sum(axis=1),
        true_theta.sum(axis=1),
        r"$\sum_{+-}\theta^*_{\ell}$",
        np.linspace(0, 2, 51),
    ),
    observable(
        pred_theta[:, 0] - pred_theta[:, 1],
        true_theta[:, 0] - true_theta[:, 1],
        r"$\Delta_{+-}\theta^*_{\ell}$",
        np.linspace(-1, 1, 61),
    ),
    observable(
        sphi(pred_phi[:, 0], pred_phi[:, 1]),
        sphi(true_phi[:, 0], true_phi[:, 1]),
        r"$\sum_{+-}\phi^*_{\ell}$",
        np.linspace(-1, 1, 61),
        log=False,
        vmax=3e2,
    ),
    observable(
        dphi(pred_phi[:, 0], pred_phi[:, 1]),
        dphi(true_phi[:, 0], true_phi[:, 1]),
        r"$\Delta_{+-}\phi^*_{\ell}$",
        np.linspace(-1, 1, 61),
        log=False,
        vmax=3e2,
    ),
]

# Mixed observables combine the theta of one lepton with the phi of either lepton.
MIXED_BINS = np.linspace(-1, 1, 51)
mixed_sum_observables = [
    observable(
        sphi(pred_theta[:, i], pred_phi[:, j]),
        sphi(true_theta[:, i], true_phi[:, j]),
        rf"$\sum_{{{CHARGES[i]}{CHARGES[j]}}}\theta^*\phi^*$",
        MIXED_BINS,
    )
    for i in range(2)
    for j in range(2)
]
mixed_diff_observables = [
    observable(
        dphi(pred_theta[:, i], pred_phi[:, j]),
        dphi(true_theta[:, i], true_phi[:, j]),
        rf"$\Delta_{{{CHARGES[i]}{CHARGES[j]}}}\theta^*\phi^*$",
        MIXED_BINS,
    )
    for i in range(2)
    for j in range(2)
]

angular_grids = [
    ("Angular sums and differences", angular_observables, False),
    ("Mixed angular sums", mixed_sum_observables, True),
    ("Mixed angular differences", mixed_diff_observables, True),
]
# The mixed grids share axes and one colorbar because their four panels share binning.
for title, observables, shared in angular_grids:
    plot_angular_1d_grid(observables, f"{title}: 1D distributions", share_axes=shared)
    plot_angular_2d_grid(
        observables, f"{title}: 2D correlations", share_axes=shared, shared_colorbar=shared
    )